In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import re
import string
from textblob import TextBlob
import nltk
from nltk.corpus import stopwords
import emoji
nltk.download('punkt')
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
nltk.download('wordnet')
nltk.download('stopwords')
from keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from keras.layers import LSTM, Dense, SimpleRNN, Embedding, Flatten, Dropout
from keras.activations import softmax
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('sentiment_tweets3.csv.zip',names=['Text','Label'],skiprows=1)
df

In [ ]:
def remove_html_tags(text):
    return BeautifulSoup(text,'html.parser').get_text()


def remove_urls(text):
    return re.sub(r'http\S+|www\S+','', text)

In [ ]:
def remove_punctuation(text):
    return text.translate(str.maketrans('','', string.punctuation))

chat_words = {
"AFAIK":"As Far As I Know",
"AFK":"Away From Keyboard",
"ASAP":"As Soon As Possible",
"ATK":"At The Keyboard",
"ATM":"At The Moment",
"A3":"Anytime, Anywhere, Anyplace",
"BAK":"Back At Keyboard",
"BBL":"Be Back Later",
"BBS":"Be Back Soon",
"BFN":"Bye For Now",
"B4N":"Bye For Now",
"BRB":"Be Right Back",
"BRT":"Be Right There",
"BTW":"By The Way",
"B4":"Before",
"B4N":"Bye For Now",
"CU":"See You",
"CUL8R":"See You Later",
"CYA":"See You",
"FAQ":"Frequently Asked Questions",
"FC":"Fingers Crossed",
"FWIW":"For What It's Worth",
"FYI":"For Your Information",
"GAL":"Get A Life",
"GG":"Good Game",
"GN":"Good Night",
"GMTA":"Great Minds Think Alike",
"GR8":"Great!",
"G9":"Genius",
"IC":"I See",
"ICQ":"I Seek you (also a chat program)",
"ILU":"ILU: I Love You",
"IMHO":"In My Honest/Humble Opinion",
"IMO":"In My Opinion",
"IOW":"In Other Words",
"IRL":"In Real Life",
"KISS":"Keep It Simple, Stupid",
"LDR":"Long Distance Relationship",
"LMAO":"Laugh My A.. Off",
"LOL":"Laughing Out Loud",
"LTNS":"Long Time No See",
"L8R":"Later",
"MTE":"My Thoughts Exactly",
"M8":"Mate",
"NRN":"No Reply Necessary",
"OIC":"Oh I See",
"PITA":"Pain In The A..",
"PRT":"Party",
"PRW":"Parents Are Watching",
"QPSA?":"Que Pasa?",
"ROFL":"Rolling On The Floor Laughing",
"ROFLOL":"Rolling On The Floor Laughing Out Loud",
"ROTFLMAO":"Rolling On The Floor Laughing My A.. Off",
"SK8":"Skate",
"STATS":"Your sex and age",
"ASL":"Age, Sex, Location",
"THX":"Thank You",
"TTFN":"Ta-Ta For Now!",
"TTYL":"Talk To You Later",
"U":"You",
"U2":"You Too",
"U4E":"Yours For Ever",
"WB":"Welcome Back",
"WTF":"What The F...",
"WTG":"Way To Go!",
"WUF":"Where Are You From?",
"W8":"Wait...",
"7K":"Sick:-D Laugher",
"TFW":"That feeling when",
"MFW":"My face when",
"MRW":"My reaction when",
"IFYP":"I feel your pain",
"TNTL":"Trying not to laugh",
"JK":"Just kidding",
"IDC":"I don't care",
"ILY":"I love you",
"IMU":"I miss you",
"ADIH":"Another day in hell",
"ZZZ":"Sleeping, bored, tired",
"WYWH":"Wish you were here",
"TIME":"Tears in my eyes",
"BAE":"Before anyone else",
"FIMH":"Forever in my heart",
"BSAAW":"Big smile and a wink",
"BWL":"Bursting with laughter",
"BFF":"Best friends forever",
"CSL":"Can't stop laughing"
}

def replace_chat_words(text):
    words = text.split()
    for i, word in enumerate(words):
        if word.lower()in chat_words:
            words[i] = chat_words[word.lower()]
    return' '.join(words)

In [ ]:
stop_words = set(stopwords.words('english'))
def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word.lower() not in stop_words]
    return' '.join(filtered_words)

def remove_emojis(text):
    return emoji.demojize(text)

In [ ]:
def preprocess(df):
    df['Text'] = df['Text'].str.lower()
    df['Text'] = df['Text'].apply(remove_html_tags)
    df['Text'] = df['Text'].apply(remove_urls)
    df['Text'] = df['Text'].apply(remove_punctuation)
    df['Text'] = df['Text'].apply(replace_chat_words)
    df['Text'] = df['Text'].apply(remove_stopwords)
    df['Text'] = df['Text'].apply(remove_emojis)
    return df

df = preprocess(df)
df

In [ ]:
X = df['Text']
y = df['Label']
y = to_categorical(y, num_classes=2)
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
str1 = ['I am going towards depression','feeling great day at work']

In [ ]:
tokenizer = Tokenizer(oov_token = 'nothing')
tokenizer.fit_on_texts(X_train)
tokenizer.fit_on_texts(X_test)
tokenizer.fit_on_texts(str1)

X_train_sequences = tokenizer.texts_to_sequences(X_train)
X_test_sequences = tokenizer.texts_to_sequences(X_test)
item_sequences = tokenizer.texts_to_sequences(str1)

In [ ]:
item_sequences

In [12]:
maxlen = max(len(tokens) for tokens in X_train_sequences)
X_train_padded = pad_sequences(X_train_sequences, maxlen=maxlen, padding='post')
X_test_padded = pad_sequences(X_test_sequences, maxlen=maxlen, padding='post')
item_padded = pad_sequences(item_sequences, maxlen=maxlen, padding='post')

item_padded

SyntaxError: invalid syntax (796671574.py, line 1)

In [ ]:
model = Sequential()
model.add(LSTM(128, input_shape=(75,1), return_sequences=True)) 
model.add(LSTM(128))
model.add(Dense(64, activation='relu'))
model.add(Dense(2, activation='softmax')) 
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy']) 
model.summary()

In [ ]:
history = model.fit(X_train_padded, y_train, epochs=5, batch_size=32, validation_data=(X_test_padded, y_test))

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper right')
plt.show()

# Plotting the training and testing loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper right')
plt.show()

In [ ]:
ypred = model.predict(X_test_padded)
ypred

In [ ]:
y_test = y_test.argmax(axis=1)

In [ ]:
from sklearn import metrics

confusion_matrix = metrics.confusion_matrix(y_test, ypred)
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix = confusion_matrix, display_labels = [0, 1])
cm_display.plot()

Accuracy = metrics.accuracy_score(y_test, ypred)
Accuracy

In [ ]:
ypred_item = model.predict(item_padded)
print(str1,ypred_item.argmax(axis=1))